# Lazada Uplift — Exploratory Data Analysis

Notebook này là bước đầu tiên của đề tài **so sánh các phương pháp ước lượng Doubly Robust CATE (DR-Learner vs DML)** cho bài toán cá nhân hóa khuyến mãi.

Ở đây **không dùng bất kỳ mô hình machine learning nào** — chỉ thống kê mô tả và trực quan hóa.

### Dữ liệu

| File | Số dòng | Bản chất |
|---|---|---|
| `full_trainset.csv` | 926.669 | Dữ liệu quan sát — voucher phát theo chiến lược vận hành |
| `full_testset.csv` | 181.669 | Dữ liệu RCT — voucher phát ngẫu nhiên |

- `is_treat` (W) — có nhận voucher hay không
- `label` (Y) — có mua hàng hay không
- `f0` … `f82` — 83 đặc trưng ẩn danh

### Bốn câu hỏi cần trả lời

1. Dữ liệu trông như thế nào, có sạch không?
2. Thiên vị chọn lọc trong tập Train lớn tới đâu?
3. Dữ liệu quan sát và dữ liệu RCT khác nhau ra sao?
4. Voucher có tác động khác nhau giữa các nhóm khách hàng không?

---
# Phần 1 — Làm quen với dữ liệu

### Bước 1 — Import thư viện

In [ ]:
import os
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
DATA_DIR = os.path.join(ROOT, 'dataset')
FIG_DIR = os.path.join(ROOT, 'reports', 'figures')
ART_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(ART_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

C_TREAT, C_CTRL, C_ACC = '#d95f02', '#2b5c8f', '#1b9e77'

def save_fig(name):
    plt.savefig(os.path.join(FIG_DIR, name + '.png'), bbox_inches='tight')

### Bước 2 — Chia dữ liệu **trước khi** nhìn vào nó

Việc đầu tiên phải làm, trước cả `head()`.

Lý do: mọi con số ta nhìn thấy đều ảnh hưởng tới quyết định sau đó — bỏ cột nào, tạo đặc trưng gì, thử mô hình nào. Nếu tập dùng để báo cáo kết quả cuối đã bị nhìn qua trong lúc khám phá, thì con số cuối cùng không còn là ước lượng trung thực cho hiệu năng trên dữ liệu mới nữa.

Cách chia:

```
full_trainset (quan sát) ──► train  80%   <- EDA được phép nhìn
                         └─► val    20%   <- để dành tinh chỉnh siêu tham số

full_testset  (RCT)      ──► rct_select  50%   <- EDA được phép nhìn
                         └─► rct_holdout 50%   <- NIÊM PHONG tới notebook 04
```

Phân chia được ghi ra file để notebook 02 dùng lại **đúng y hệt** — chia hai lần bằng hai đoạn code khác nhau là mầm mống của sai lệch âm thầm.

Chia phân tầng theo `(is_treat, label)` vì tỉ lệ mua hàng chỉ khoảng 2%, chia ngẫu nhiên thuần dễ làm lệch tỉ lệ giữa các tập.

In [ ]:
from sklearn.model_selection import train_test_split

train_full = pd.read_csv(os.path.join(DATA_DIR, 'full_trainset.csv'))
test_full = pd.read_csv(os.path.join(DATA_DIR, 'full_testset.csv'))

FEATURES = [f'f{i}' for i in range(83)]
for df in (train_full, test_full):
    df[FEATURES] = df[FEATURES].astype('float32')

def make_strata(df):
    return df['is_treat'].astype(str) + '_' + df['label'].astype(str)

tr_idx, val_idx = train_test_split(
    train_full.index, test_size=0.20, random_state=SEED,
    stratify=make_strata(train_full))

sel_idx, hold_idx = train_test_split(
    test_full.index, test_size=0.50, random_state=SEED,
    stratify=make_strata(test_full))

assignment = pd.concat([
    pd.DataFrame({'data_id': train_full.loc[tr_idx, 'data_id'], 'split': 'train'}),
    pd.DataFrame({'data_id': train_full.loc[val_idx, 'data_id'], 'split': 'val'}),
    pd.DataFrame({'data_id': test_full.loc[sel_idx, 'data_id'], 'split': 'rct_select'}),
    pd.DataFrame({'data_id': test_full.loc[hold_idx, 'data_id'], 'split': 'rct_holdout'}),
], ignore_index=True)

assign_path = os.path.join(ART_DIR, 'split_assignment.parquet')
assignment.to_parquet(assign_path, index=False)

print('Đã ghi phân chia:', assign_path)
display(assignment['split'].value_counts().to_frame('Số dòng'))

### Bước 3 — Giữ lại đúng phần được phép nhìn

Từ đây tới hết notebook, chỉ tồn tại hai bảng: `train` và `rct`. Hai tập còn lại được xóa khỏi bộ nhớ ngay, để không thể vô tình dùng nhầm.

In [ ]:
train = train_full.loc[tr_idx].reset_index(drop=True)
rct = test_full.loc[sel_idx].reset_index(drop=True)

del train_full, test_full          # không để lộ val và rct_holdout ở bất kỳ đâu phía sau

print(f'train      : {train.shape[0]:>8,} dòng  (80% dữ liệu quan sát)')
print(f'rct_select : {rct.shape[0]:>8,} dòng  (50% dữ liệu RCT)')
print()
print('val và rct_holdout: KHÔNG nạp vào notebook này.')

### Bước 4 — Xem 5 dòng đầu

In [ ]:
train.head()

### Bước 5 — Xem 5 dòng cuối

In [ ]:
train.tail()

### Bước 6 — Tập RCT-select trông có giống không?

In [ ]:
rct.head()

### Bước 7 — Thông tin tổng quan các cột

In [ ]:
train.info()

### Bước 8 — `data_id` có bị trùng không?

Nếu trùng thì một khách hàng xuất hiện nhiều lần, sẽ ảnh hưởng tới cách chia dữ liệu sau này.

In [ ]:
print('Số data_id trùng trong Train:', train['data_id'].duplicated().sum())
print('Số data_id trùng trong RCT-select:', rct['data_id'].duplicated().sum())
print('\nVí dụ vài data_id:', train['data_id'].head(3).tolist())

### Bước 9 — Giá trị khuyết thiếu

In [ ]:
print('Tổng ô khuyết thiếu — Train:', train.isna().sum().sum())
print('Tổng ô khuyết thiếu — RCT-select:', rct.isna().sum().sum())

na_cols = train.isna().sum()
print('\nSố cột có giá trị khuyết thiếu:', (na_cols > 0).sum())

### Bước 10 — Thống kê mô tả của hai biến chính

In [ ]:
train[['is_treat', 'label']].describe().T

### Bước 11 — Thống kê mô tả toàn bộ 83 đặc trưng

Đây là bảng mentor yêu cầu đưa vào slide: phân phối và thang đo của từng biến, để biết bước kỹ nghệ đặc trưng sau này có cần chuẩn hóa hay không.

In [ ]:
desc = train[FEATURES].describe().T
desc['range'] = desc['max'] - desc['min']
desc.round(2)

Các đặc trưng có thang đo rất khác nhau — có cột chỉ chạy trong khoảng 0–1, có cột lên tới hàng trăm hoặc hàng nghìn.

Với mô hình cây quyết định (LightGBM) thì **không cần chuẩn hóa**, vì cây chia theo ngưỡng chứ không theo khoảng cách. Đây là lý do notebook 02 sẽ bỏ qua bước standardize.

### Bước 12 — Mỗi cột có bao nhiêu giá trị khác nhau?

Dữ liệu ẩn danh hoàn toàn nên không biết tên nghiệp vụ. Cách duy nhất để đoán bản chất từng cột là đếm số giá trị khác nhau.

In [ ]:
nunique = train[FEATURES].nunique().sort_values()
nunique.head(15).to_frame('Số giá trị khác nhau')

### Bước 13 — Phân loại kiểu đặc trưng

In [ ]:
def classify(n):
    if n == 1:
        return 'Hằng số'
    if n == 2:
        return 'Nhị phân'
    if n <= 10:
        return 'Rời rạc'
    return 'Liên tục'

col_kind = nunique.apply(classify)
kind_count = col_kind.value_counts()
print(kind_count.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
kind_count.plot(kind='barh', ax=ax, color=[C_CTRL, C_ACC, C_TREAT, '#999999'][:len(kind_count)])
ax.set_title('Phân loại 83 đặc trưng theo số giá trị khác nhau')
ax.set_xlabel('Số cột')
for i, v in enumerate(kind_count.values):
    ax.text(v + 0.5, i, str(v), va='center')
plt.tight_layout()
save_fig('01_phan_loai_dac_trung')
plt.show()

Gần một nửa số cột là nhị phân hoặc rời rạc. Điều này quyết định một chi tiết ở notebook 04: khi tính độ quan trọng đặc trưng **phải dùng `importance_type='gain'`**. Mặc định của LightGBM là đếm số lần chia nhánh, mà cột nhị phân chỉ chia được một lần nên luôn bị đánh giá thấp một cách oan uổng.

### Bước 14 — Cột hằng số

Cột chỉ có một giá trị duy nhất thì không mang thông tin gì.

In [ ]:
constant_cols = sorted(nunique[nunique == 1].index.tolist())
print('Cột hằng số:', constant_cols)

for c in constant_cols:
    print(f'  {c} = {train[c].iloc[0]} (toàn bộ {len(train):,} dòng)')

### Bước 15 — Cột trùng lặp

Hai cột giống hệt nhau từng dòng thì giữ cả hai là thừa. Tệ hơn, mô hình cây sẽ chia đôi độ quan trọng cho cùng một thông tin, khiến bảng feature importance sau này bị sai lệch.

Cách tìm: băm nội dung từng cột rồi so mã băm — nhanh hơn nhiều so với so từng cặp một.

In [ ]:
seen = {}
duplicate_map = {}          # cột bị bỏ -> cột được giữ
for c in FEATURES:
    key = hashlib.md5(np.ascontiguousarray(train[c].values)).hexdigest()
    if key in seen:
        duplicate_map[c] = seen[key]
    else:
        seen[key] = c

groups = {}
for dup, keep in duplicate_map.items():
    groups.setdefault(keep, []).append(dup)

print(f'Tìm thấy {len(duplicate_map)} cột trùng lặp:\n')
for keep, dups in sorted(groups.items(), key=lambda x: int(x[0][1:])):
    print(f'  Giữ {keep:<5} — bỏ {", ".join(dups)}')

> **Lưu ý:** với mỗi nhóm cột trùng nhau phải **giữ lại một cột** và chỉ bỏ những cột còn lại. Bỏ hết cả nhóm là mất trắng thông tin đó.

### Bước 16 — Chốt danh sách cột loại bỏ

In [ ]:
COLS_TO_DROP = sorted(set(constant_cols) | set(duplicate_map.keys()),
                      key=lambda c: int(c[1:]))
CLEAN_FEATURES = [c for c in FEATURES if c not in COLS_TO_DROP]
n_feat = len(CLEAN_FEATURES)

print('Loại bỏ  :', len(COLS_TO_DROP), 'cột ->', COLS_TO_DROP)
print('Giữ lại  :', n_feat, 'đặc trưng sạch')

### Tóm tắt Phần 1

- Dữ liệu **không có giá trị khuyết thiếu** — đã qua xử lý ETL sẵn, bỏ qua được khâu điền khuyết.
- `data_id` không trùng, mỗi dòng là một bản ghi độc lập.
- Các đặc trưng có thang đo rất khác nhau, nhưng dùng mô hình cây nên không cần chuẩn hóa.
- Danh sách cột loại bỏ được **tính ra từ dữ liệu** chứ không gõ tay, nên không sợ bỏ sót hay bỏ nhầm.

---
# Phần 2 — Treatment (W) và Outcome (Y)

### Bước 17 — Bao nhiêu người được phát voucher?

In [ ]:
print('--- TRAIN ---')
print(train['is_treat'].value_counts())
print(train['is_treat'].value_counts(normalize=True).round(4))

print('\n--- TEST ---')
print(rct['is_treat'].value_counts())
print(rct['is_treat'].value_counts(normalize=True).round(4))

Chênh lệch rất lớn: tập Train chỉ khoảng 22% người nhận voucher, còn tập RCT gần như chia đôi.

Đây là dấu hiệu đầu tiên cho thấy hai tập đến từ hai cơ chế khác nhau — dữ liệu quan sát từ vận hành thực tế, dữ liệu RCT từ một thí nghiệm ngẫu nhiên.

### Bước 18 — Bao nhiêu người mua hàng?

In [ ]:
print('--- TRAIN ---')
print(train['label'].value_counts())
print(train['label'].value_counts(normalize=True).round(4))

print('\n--- TEST ---')
print(rct['label'].value_counts(normalize=True).round(4))

Chỉ khoảng 2% người mua hàng trên tập Train. Mất cân bằng lớp rất nặng, kéo theo ba hệ quả:

- Chia dữ liệu bắt buộc phải **stratify**
- Các chỉ số đánh giá phải dùng nhóm phân vị (Qini, AUUC) thay vì độ chính xác thông thường
- Cần cỡ mẫu lớn mới đo chính xác được hiệu ứng nhỏ

### Bước 19 — Bảng chéo W × Y

In [ ]:
print('--- TRAIN ---')
display(pd.crosstab(train['is_treat'], train['label'], margins=True))

print('--- TEST (RCT) ---')
display(pd.crosstab(rct['is_treat'], rct['label'], margins=True))

### Bước 20 — Tỉ lệ chuyển đổi theo từng nhóm

In [ ]:
cvr_train = train.groupby('is_treat')['label'].mean()
cvr_rct = rct.groupby('is_treat')['label'].mean()

cvr = pd.DataFrame({'Train (quan sát)': cvr_train, 'RCT-select': cvr_rct})
cvr.index = ['Control (W=0)', 'Treatment (W=1)']
display(cvr.round(4))

print(f'Chênh lệch Train: {(cvr_train[1] - cvr_train[0])*100:.3f} điểm phần trăm')
print(f'Chênh lệch RCT-select: {(cvr_rct[1] - cvr_rct[0])*100:.3f} điểm phần trăm')

Hai con số chênh lệch này khác nhau một trời một vực, dù cùng đo một thứ. Bước 21 sẽ mổ xẻ tại sao.

### Bước 21 — Trực quan hóa

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

w_rates = pd.DataFrame({
    'Control (W=0)': [1 - train.is_treat.mean(), 1 - rct.is_treat.mean()],
    'Treatment (W=1)': [train.is_treat.mean(), rct.is_treat.mean()],
}, index=['Train', 'RCT-select'])
w_rates.plot(kind='bar', stacked=True, ax=axes[0], color=[C_CTRL, C_TREAT], width=0.6)
axes[0].set_title('Tỉ lệ phát voucher')
axes[0].tick_params(axis='x', rotation=0)
axes[0].axhline(0.5, color='black', ls=':', lw=1)
axes[0].legend(fontsize=8)

cvr.T.plot(kind='bar', ax=axes[1], color=[C_CTRL, C_TREAT], width=0.7)
axes[1].set_title('Tỉ lệ chuyển đổi theo nhóm')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=8)
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt='%.4f', fontsize=8)

axes[2].bar(['Không mua (Y=0)', 'Có mua (Y=1)'],
            [(train.label == 0).mean(), (train.label == 1).mean()],
            color=['#cccccc', C_ACC], width=0.6)
axes[2].set_title('Mất cân bằng lớp (Train)')
for i, v in enumerate([(train.label == 0).mean(), (train.label == 1).mean()]):
    axes[2].text(i, v + 0.02, f'{v*100:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
save_fig('02_phan_phoi_treatment_outcome')
plt.show()

### Bước 22 — Ước lượng ngây thơ so với sự thật

Đây là phép so sánh quan trọng nhất của cả notebook.

Trên tập **RCT-select**, voucher phát ngẫu nhiên nên hai nhóm tương đương nhau về mọi mặt. Hiệu hai tỉ lệ chuyển đổi chính là **hiệu ứng nhân quả thật**.

Trên tập **Train (quan sát)**, voucher được nhắm vào những người vốn đã có xu hướng mua. Lấy hiệu hai tỉ lệ chuyển đổi sẽ ra một con số trộn lẫn giữa tác động thật của voucher và việc nhóm được chọn vốn đã khác biệt sẵn.

In [ ]:
ate_naive = cvr_train[1] - cvr_train[0]

t_grp = rct.loc[rct.is_treat == 1, 'label']
c_grp = rct.loc[rct.is_treat == 0, 'label']
p1, p0, n1, n0 = t_grp.mean(), c_grp.mean(), len(t_grp), len(c_grp)
ate_rct = p1 - p0
se_rct = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
ci_lo, ci_hi = ate_rct - 1.96 * se_rct, ate_rct + 1.96 * se_rct

print('Ngây thơ (dữ liệu quan sát):', f'{ate_naive*100:.3f} điểm phần trăm')
print('Sự thật  (RCT)             :', f'{ate_rct*100:.3f} điểm phần trăm',
      f'(KTC 95%: {ci_lo*100:.3f} — {ci_hi*100:.3f})')
print()
print(f'=> Ước lượng ngây thơ thổi phồng hiệu ứng lên {ate_naive/ate_rct:.1f} LẦN')

### Bước 23 — Trực quan hóa mức sai lệch

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Ước lượng ngây thơ\n(dữ liệu quan sát)', 'Hiệu ứng thật\n(RCT)'],
              [ate_naive * 100, ate_rct * 100],
              color=['#c0392b', C_ACC], width=0.5)
ax.errorbar(1, ate_rct * 100, yerr=1.96 * se_rct * 100,
            fmt='none', ecolor='black', capsize=8, lw=1.5)
ax.set_ylabel('Mức tăng tỉ lệ chuyển đổi (điểm phần trăm)')
ax.set_title(f'Thiên vị chọn lọc thổi phồng hiệu ứng lên {ate_naive/ate_rct:.1f} lần')

tops = [ate_naive * 100, ate_rct * 100 + 1.96 * se_rct * 100]
for b, v, top in zip(bars, [ate_naive * 100, ate_rct * 100], tops):
    ax.text(b.get_x() + b.get_width() / 2, top + 0.15, f'{v:.3f} pp',
            ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0, ate_naive * 100 * 1.25)
plt.tight_layout()
save_fig('03_ate_ngay_tho_vs_rct')
plt.show()

### Tóm tắt Phần 2

Ước lượng ngây thơ sai lệch hơn một chục lần so với hiệu ứng thật. Nguyên nhân: hệ thống vận hành đã nhắm voucher vào đúng những người vốn dĩ đã có ý định mua, nên phần lớn con số đó là **thiên vị chọn lọc** chứ không phải công của voucher.

Đây chính là lý do đề tài cần tới các phương pháp doubly robust. Chúng chỉnh lại thiên vị bằng cách kết hợp mô hình xác suất nhận voucher (propensity) với mô hình dự đoán kết quả (outcome) — chỉ cần **một trong hai** mô hình đúng là ước lượng vẫn không chệch.

---
# Phần 3 — Đo thiên vị chọn lọc

Phần 2 cho thấy *có* thiên vị. Phần này đi tìm thiên vị nằm ở **đặc trưng nào** và **lớn tới đâu**.

### Bước 24 — So sánh trung bình hai nhóm

Bắt đầu đơn giản nhất: lấy trung bình từng đặc trưng theo nhóm rồi nhìn thử.

In [ ]:
cmp = train.groupby('is_treat')[CLEAN_FEATURES[:12]].mean().T
cmp.columns = ['Control (W=0)', 'Treatment (W=1)']
cmp['Chênh lệch'] = cmp['Treatment (W=1)'] - cmp['Control (W=0)']
cmp.round(3)

Thấy rõ là hai nhóm khác nhau, nhưng chưa so sánh được giữa các đặc trưng với nhau vì mỗi cột một thang đo. Chênh lệch 100 đơn vị ở cột chạy tới hàng nghìn thì nhỏ, nhưng ở cột chạy 0–1 thì là khổng lồ.

Cần chuẩn hóa lại — đó là việc của SMD ở bước sau.

### Bước 25 — Standardized Mean Difference (SMD)

Chênh lệch trung bình chia cho độ lệch chuẩn gộp:

$$\text{SMD} = \frac{\bar{X}_{W=1} - \bar{X}_{W=0}}{\sqrt{(s^2_{W=1} + s^2_{W=0})/2}}$$

Quy ước đọc kết quả trong tài liệu nhân quả:

| \|SMD\| | Ý nghĩa |
|---|---|
| < 0,10 | Cân bằng tốt |
| 0,10 – 0,25 | Lệch nhẹ |
| > 0,25 | **Lệch nghiêm trọng**, cần hiệu chỉnh |

In [ ]:
def compute_smd(df_a, df_b, cols):
    mean_a, mean_b = df_a[cols].mean(), df_b[cols].mean()
    var_a, var_b = df_a[cols].var(), df_b[cols].var()
    pooled_sd = np.sqrt((var_a + var_b) / 2)
    return ((mean_a - mean_b) / pooled_sd.replace(0, np.nan)).fillna(0.0)

smd_train = compute_smd(train[train.is_treat == 1], train[train.is_treat == 0],
                        CLEAN_FEATURES).abs().sort_values(ascending=False)

smd_train.head(10).to_frame('|SMD| Train').round(3)

### Bước 26 — Kiểm chứng công thức bằng tập RCT

Tính lại đúng công thức đó trên tập RCT-select. Vì tập này đã ngẫu nhiên hóa nên SMD **phải gần 0**. Nếu ra số lớn thì công thức viết sai.

In [ ]:
smd_rct = compute_smd(rct[rct.is_treat == 1], rct[rct.is_treat == 0],
                       CLEAN_FEATURES).abs()

print('SMD lớn nhất — Train:', round(smd_train.max(), 3))
print('SMD lớn nhất — RCT-select:', round(smd_rct.max(), 3), '  <- gần 0, công thức đúng')
print()

bang = pd.DataFrame({
    'Train (quan sát)': [((smd_train >= lo) & (smd_train < hi)).sum()
                         for lo, hi in [(0, .1), (.1, .25), (.25, np.inf)]],
    'RCT-select': [((smd_rct >= lo) & (smd_rct < hi)).sum()
                   for lo, hi in [(0, .1), (.1, .25), (.25, np.inf)]],
}, index=['Cân bằng tốt (<0,10)', 'Lệch nhẹ (0,10–0,25)', 'Lệch nghiêm trọng (>0,25)'])
display(bang)

### Bước 27 — Bảng so sánh trực tiếp top 10

In [ ]:
pd.DataFrame({
    '|SMD| Train': smd_train.head(10),
    '|SMD| RCT-select': smd_rct[smd_train.head(10).index],
}).round(3)

### Bước 28 — Love plot

In [ ]:
top20 = smd_train.head(20).index[::-1]
y_pos = np.arange(len(top20))

fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(y_pos, smd_rct[top20], smd_train[top20], color='#bbbbbb', lw=1.5, zorder=1)
ax.scatter(smd_train[top20], y_pos, s=90, color='#c0392b', label='Train (quan sát)', zorder=3)
ax.scatter(smd_rct[top20], y_pos, s=90, color=C_ACC, label='RCT-select', zorder=3, marker='D')
ax.axvline(0.10, color='gray', ls='--', lw=1)
ax.axvline(0.25, color='#c0392b', ls='--', lw=1.2)
ax.text(0.26, 0.4, 'ngưỡng lệch nghiêm trọng 0,25', color='#c0392b', fontsize=9)
ax.set_yticks(y_pos)
ax.set_yticklabels(top20)
ax.set_xlabel('|SMD|')
ax.set_title('20 đặc trưng thiên vị nhất\nRCT gần như cân bằng, dữ liệu quan sát thì không')
ax.legend(loc='lower right')
plt.tight_layout()
save_fig('04_love_plot_smd')
plt.show()

### Bước 29 — Nhìn phân phối thực tế của 4 đặc trưng lệch nhất

In [ ]:
top4 = smd_train.head(4).index.tolist()
sample = train.sample(n=min(80_000, len(train)), random_state=SEED)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.ravel(), top4):
    lo, hi = sample[col].quantile([0.005, 0.995])
    bins = np.linspace(lo, hi, 45) if hi > lo else 30
    ax.hist(sample.loc[sample.is_treat == 0, col], bins=bins, alpha=0.55,
            color=C_CTRL, label='Control (W=0)', density=True)
    ax.hist(sample.loc[sample.is_treat == 1, col], bins=bins, alpha=0.55,
            color=C_TREAT, label='Treatment (W=1)', density=True)
    ax.set_title(f'{col}  —  |SMD| = {smd_train[col]:.2f}')
    ax.legend(fontsize=9)

plt.suptitle('Hai nhóm là hai tập khách hàng khác hẳn nhau (Train)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('05_phan_phoi_top4_thien_vi')
plt.show()

### Tóm tắt Phần 3

Trên **tập Train**, phần lớn đặc trưng vượt ngưỡng lệch nghiêm trọng, nhóm đầu bảng có SMD xấp xỉ 0,9 — trung bình hai nhóm cách nhau gần trọn một độ lệch chuẩn. Nói đơn giản: người được phát voucher và người không được phát là **hai tập khách hàng khác hẳn nhau**.

Trên **tập RCT-select**, mọi đặc trưng đều nằm dưới ngưỡng cân bằng tốt. Phép ngẫu nhiên hóa đã làm đúng việc của nó, và kết quả này đồng thời xác nhận công thức SMD viết đúng.

Đối chiếu hai kết quả là **bằng chứng trực tiếp** cho lựa chọn phương pháp của đề tài: với dữ liệu lệch tới mức này, mô hình dự đoán thông thường sẽ học nhầm "đặc điểm của người được chọn phát voucher" thành "đặc điểm của người phản ứng tốt với voucher".

Biểu đồ ở bước 28 cũng cho thấy hai nhóm vẫn **chồng lấn** chứ không tách rời hẳn — giả định positivity còn giữ được, nên vẫn ước lượng được hiệu ứng nhân quả.

---
# Phần 4 — Dữ liệu quan sát và dữ liệu RCT có cùng một quần thể không?

Phần 3 so sánh Treatment với Control *bên trong* từng tập. Phần này so sánh **tập train với tập rct_select**, để biết mô hình huấn luyện trên dữ liệu quan sát có áp được lên dữ liệu RCT hay không.

### Bước 30 — Đo bằng chính công cụ SMD

In [ ]:
shift = compute_smd(train, rct, CLEAN_FEATURES).abs().sort_values(ascending=False)

print('Dịch chuyển lớn nhất:', round(shift.max(), 3), f'({shift.index[0]})')
print()
print('Lệch nghiêm trọng (>0,25):', (shift > 0.25).sum(), '/', n_feat)
print('Lệch nhẹ (0,10–0,25)     :', ((shift >= 0.10) & (shift <= 0.25)).sum(), '/', n_feat)
print('Cân bằng tốt (<0,10)     :', (shift < 0.10).sum(), '/', n_feat)

### Bước 31 — 15 đặc trưng dịch chuyển mạnh nhất

In [ ]:
top15 = shift.head(15)[::-1]
colors = ['#c0392b' if v > 0.25 else ('#e8a33d' if v > 0.10 else C_ACC) for v in top15]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15.index, top15.values, color=colors)
ax.axvline(0.10, color='gray', ls='--', lw=1)
ax.axvline(0.25, color='#c0392b', ls='--', lw=1.2)
ax.set_xlabel('|SMD| giữa train và rct_select')
ax.set_title('Dịch chuyển phân phối giữa hai môi trường dữ liệu')
plt.tight_layout()
save_fig('06_covariate_shift')
plt.show()

### Bước 32 — Nhìn phân phối thực tế

In [ ]:
top3_shift = shift.head(3).index.tolist()
s_tr = train.sample(n=min(60_000, len(train)), random_state=SEED)
s_te = rct.sample(n=min(60_000, len(rct)), random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, top3_shift):
    lo = min(s_tr[col].quantile(0.005), s_te[col].quantile(0.005))
    hi = max(s_tr[col].quantile(0.995), s_te[col].quantile(0.995))
    bins = np.linspace(lo, hi, 45) if hi > lo else 30
    ax.hist(s_tr[col], bins=bins, alpha=0.55, color=C_CTRL, label='Train', density=True)
    ax.hist(s_te[col], bins=bins, alpha=0.55, color=C_ACC, label='RCT-select', density=True)
    ax.set_title(f'{col}  —  |SMD| = {shift[col]:.2f}')
    ax.legend(fontsize=9)

plt.tight_layout()
save_fig('07_phan_phoi_train_vs_test')
plt.show()

### Tóm tắt Phần 4

Có dịch chuyển phân phối giữa hai tập, nhưng **nhẹ hơn nhiều** so với mức lệch giữa Treatment và Control ở Phần 3. Hai tập vẫn là cùng một quần thể khách hàng, chỉ khác nhau ở cách chọn mẫu.

Điều này hợp thức hóa thiết kế thực nghiệm: huấn luyện trên dữ liệu quan sát rồi đánh giá trên dữ liệu RCT là hợp lệ. Nhưng cần lưu ý khi diễn giải — nếu mô hình đạt điểm thấp trên RCT thì có hai khả năng, hoặc mô hình dở, hoặc do dịch chuyển phân phối.

Nhóm dịch chuyển mạnh nhất trùng khá nhiều với nhóm thiên vị mạnh nhất ở Phần 3. Hợp lý, vì đó chính là những đặc trưng mà hệ thống vận hành dùng để quyết định phát voucher cho ai.

---
# Phần 5 — Cấu trúc tương quan

### Bước 33 — Đặc trưng nào liên quan tới việc được phát voucher?

In [ ]:
sample_corr = train.sample(n=min(150_000, len(train)), random_state=SEED)

corr_w = sample_corr[CLEAN_FEATURES].corrwith(sample_corr['is_treat']).sort_values(
    key=abs, ascending=False)
corr_y = sample_corr[CLEAN_FEATURES].corrwith(sample_corr['label']).sort_values(
    key=abs, ascending=False)

pd.DataFrame({
    'Tương quan với W (nhận voucher)': corr_w.head(10),
    'Tương quan với Y (mua hàng)': corr_y[corr_w.head(10).index],
}).round(3)

Những đặc trưng liên quan mạnh tới `W` **cũng** liên quan tới `Y`. Đây đúng định nghĩa của **confounder**: biến vừa ảnh hưởng tới việc ai được phát voucher, vừa ảnh hưởng tới việc ai mua hàng.

Chính chúng tạo ra khoảng sai lệch hơn chục lần đã thấy ở Phần 2.

### Bước 34 — Đặc trưng nào liên quan tới hành vi mua hàng?

In [ ]:
corr_y.head(10).to_frame('Tương quan với Y').round(3)

### Bước 35 — Các cặp đặc trưng tương quan cao

In [ ]:
cm = sample_corr[CLEAN_FEATURES].corr()
pairs = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool)).stack()

high_pairs = pairs[pairs.abs() > 0.90].sort_values(key=abs, ascending=False)
print('Số cặp có |tương quan| > 0,90:', len(high_pairs))
display(high_pairs.head(12).to_frame('Hệ số tương quan').round(3))

### Bước 36 — Cặp tương quan tuyệt đối bằng 1

Đây là những cột mà một cái là biến đổi tuyến tính của cái kia — kiểu `x` và `-x`, hoặc `x` và `1-x`. Phép băm ở bước 14 không bắt được vì giá trị cụ thể khác nhau, nhưng về mặt thông tin thì chúng **thừa hoàn toàn**.

Riêng phép dò này tính trên **toàn bộ tập train**, không dùng mẫu con như các bước trên. Lý do: đây là phép so đẳng thức chính xác, mà một cặp có thể đạt đúng 1 trên mẫu vài trăm nghìn dòng nhưng chỉ còn 0,9999 trên toàn tập — và ngược lại. Danh sách này được notebook 02 dùng để loại cột nên phải ổn định, không được đổi theo cách lấy mẫu.

In [ ]:
cm_full = train[CLEAN_FEATURES].corr()
pairs_full = cm_full.where(
    np.triu(np.ones(cm_full.shape), k=1).astype(bool)).stack()

perfect = pairs_full[pairs_full.abs().round(6) >= 1.0].sort_index()
print('Số cặp tương quan tuyệt đối (tính trên toàn bộ train):', len(perfect))
display(perfect.to_frame('Hệ số tương quan').round(4))

Các cặp này nối với nhau thành **nhóm** — nếu `a` trùng `b` và `b` trùng `c` thì cả ba cùng một nhóm. Gom nhóm rồi mỗi nhóm giữ đúng một cột.

In [ ]:
# Gom nhóm bắc cầu: a~b, b~c  =>  {a, b, c}
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb, key=lambda c: int(c[1:]))] = min(ra, rb, key=lambda c: int(c[1:]))

for a, b in perfect.index:
    union(a, b)

linear_groups = {}
for c in parent:
    linear_groups.setdefault(find(c), []).append(c)

LINEAR_REDUNDANT = []
print('Các nhóm cột thừa tuyến tính:')
for keep, members in sorted(linear_groups.items(), key=lambda x: int(x[0][1:])):
    members = sorted(members, key=lambda c: int(c[1:]))
    drop = [c for c in members if c != keep]
    LINEAR_REDUNDANT += drop
    print(f'  Nhóm {members} -> giữ {keep}, bỏ {drop}')

LINEAR_REDUNDANT = sorted(set(LINEAR_REDUNDANT), key=lambda c: int(c[1:]))
print(f'\nTổng cột thừa tuyến tính cần bỏ thêm: {LINEAR_REDUNDANT}')

### Bước 37 — Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(sample_corr[smd_train.head(20).index.tolist()].corr(),
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.4, cbar_kws={'shrink': 0.75}, ax=ax)
ax.set_title('Ma trận tương quan — 20 đặc trưng thiên vị nhất')
plt.tight_layout()
save_fig('08_heatmap_tuong_quan')
plt.show()

### Tóm tắt Phần 5

Nhóm đặc trưng tương quan mạnh nhất với `is_treat` chính là nhóm có SMD cao nhất ở Phần 3 — hai phép đo độc lập cùng chỉ về một kết luận.

Heatmap cho thấy chúng tụ thành cụm tương quan cao với nhau. Mô hình cây chịu được đa cộng tuyến nên không bắt buộc cắt bớt, nhưng cần nhớ khi đọc bảng feature importance ở notebook 04: trong một cụm tương quan cao, việc mô hình chọn cột nào gần như là ngẫu nhiên.

---
# Phần 6 — Voucher có tác động khác nhau giữa các nhóm không?

Đây là câu hỏi sống còn của đề tài. Nếu voucher tác động **như nhau** tới mọi khách hàng thì không cần mô hình CATE — cứ phát cho tất cả hoặc không phát cho ai.

Cách kiểm tra không cần mô hình: chia khách hàng thành các nhóm theo phân vị của một đặc trưng, rồi trong từng nhóm tính chênh lệch tỉ lệ chuyển đổi giữa người có và không có voucher.

**Bắt buộc làm trên tập RCT-select.** Trên tập Train, chênh lệch trong từng nhóm vẫn dính thiên vị chọn lọc nên vô nghĩa.

### Bước 38 — Hàm tính uplift theo phân vị

In [ ]:
def subgroup_uplift(df, col, n_bins=5, min_size=200):
    try:
        bins = pd.qcut(df[col], n_bins, labels=False, duplicates='drop')
    except ValueError:
        return pd.DataFrame()

    rows = []
    for b in sorted(pd.Series(bins).dropna().unique()):
        sub = df[bins == b]
        t = sub.loc[sub.is_treat == 1, 'label']
        c = sub.loc[sub.is_treat == 0, 'label']
        if len(t) < min_size or len(c) < min_size:
            continue
        pt, pc = t.mean(), c.mean()
        se = np.sqrt(pt * (1 - pt) / len(t) + pc * (1 - pc) / len(c))
        rows.append({'Nhóm': f'Q{int(b)+1}', 'Số mẫu': len(sub),
                     'CVR Treatment': pt, 'CVR Control': pc,
                     'Uplift': pt - pc, 'Biên KTC 95%': 1.96 * se})
    return pd.DataFrame(rows)

### Bước 39 — Thử với đặc trưng thiên vị nhất

In [ ]:
col0 = [c for c in smd_train.index if nunique[c] > 20][0]
print('Phân vị theo', col0, '— trên tập RCT\n')
subgroup_uplift(rct, col0).round(4)

Uplift **không giống nhau** giữa các nhóm. Chạy tiếp với 4 đặc trưng để xem có phải chỉ là ngẫu nhiên hay không.

### Bước 40 — Chạy với 4 đặc trưng liên tục thiên vị nhất

In [ ]:
continuous = [c for c in smd_train.index if nunique[c] > 20][:4]

results = {}
for col in continuous:
    r = subgroup_uplift(rct, col)
    if not r.empty:
        results[col] = r

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4.8), squeeze=False)

for ax, (col, r) in zip(axes[0], results.items()):
    ax.bar(r['Nhóm'], r['Uplift'] * 100, yerr=r['Biên KTC 95%'] * 100,
           color=C_ACC, capsize=4, width=0.65, error_kw={'lw': 1.2})
    ax.axhline(ate_rct * 100, color='black', ls='--', lw=1.3,
               label=f'ATE chung = {ate_rct*100:.2f} pp')
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_title(f'Phân vị theo {col}')
    ax.set_ylabel('Uplift (điểm phần trăm)')
    ax.legend(fontsize=8)

plt.suptitle('Hiệu ứng voucher thay đổi theo phân khúc khách hàng (RCT)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('09_uplift_theo_phan_khuc')
plt.show()

### Bước 41 — Biên độ chênh lệch so với mức trung bình

In [ ]:
spread = {c: r['Uplift'].max() - r['Uplift'].min() for c, r in results.items()}

pd.DataFrame([
    {'Đặc trưng': c,
     'Chênh lệch cao nhất - thấp nhất (pp)': round(v * 100, 3),
     'Gấp bao nhiêu lần ATE chung': round(v / abs(ate_rct), 1)}
    for c, v in sorted(spread.items(), key=lambda x: -x[1])
])

### Tóm tắt Phần 6

Uplift **không đồng đều** giữa các phân khúc. Ở mọi đặc trưng khảo sát đều có những nhóm mà hiệu ứng gần như bằng 0 — khoảng tin cậy phủ qua mốc 0, tức chưa đủ bằng chứng khẳng định voucher có tác dụng — bên cạnh những nhóm hiệu ứng rõ rệt và lớn gấp nhiều lần mức trung bình.

Biên độ chênh lệch lớn gấp khoảng **1,6 đến 2,7 lần** chính giá trị ATE trung bình. Nói cách khác, **con số trung bình che giấu phần lớn thông tin hữu ích**.

Hình dạng đường uplift cũng khác nhau tùy đặc trưng: có đặc trưng cho quan hệ giảm dần đều, có đặc trưng lại đạt đỉnh ở nhóm giữa. Kiểu quan hệ phi tuyến này là lý do cần tới NonParamDML và CausalForestDML chứ không thể chỉ dùng LinearDML.

Về mặt nghiệp vụ, đây chính là mục tiêu bài toán: tìm ra nhóm **Persuadables** — người chỉ mua khi có voucher — và tránh tiêu ngân sách vào nhóm vốn dĩ đã mua sẵn.

Hai lưu ý khi đọc biểu đồ:

- Một số đặc trưng chỉ hiện 3–4 nhóm thay vì 5, do có nhiều bản ghi trùng giá trị ở đuôi phân phối nên ranh giới phân vị bị gộp lại. Đây là hành vi đúng, không phải lỗi.
- Khoảng tin cậy khá rộng vì tỉ lệ chuyển đổi chỉ khoảng 3%.

---
# Phần 7 — Tổng hợp & Bàn giao

### Bước 42 — Bảng tổng hợp kết quả

In [ ]:
findings = pd.DataFrame([
    ('Quy mô', f'{len(train):,} dòng quan sát + {len(rct):,} dòng RCT, 83 đặc trưng ẩn danh'),
    ('Chất lượng', f'0 khuyết thiếu, 0 data_id trùng; loại {len(COLS_TO_DROP)} cột '
                   f'(hằng số + trùng lặp), còn {n_feat} đặc trưng sạch'),
    ('Kiểu đặc trưng', ' · '.join(f'{k}: {v}' for k, v in kind_count.items())),
    ('Tỉ lệ phát voucher', f'train {train.is_treat.mean()*100:.2f}% vs rct_select {rct.is_treat.mean()*100:.2f}%'),
    ('Mất cân bằng lớp', f'Chỉ {train.label.mean()*100:.2f}% mua hàng -> bắt buộc chia stratify'),
    ('Thiên vị chọn lọc', f'{(smd_train > 0.25).sum()}/{n_feat} đặc trưng SMD > 0,25 '
                          f'(cao nhất {smd_train.max():.2f})'),
    ('Kiểm chứng RCT', f'SMD lớn nhất trên rct_select chỉ {smd_rct.max():.3f} -> ngẫu nhiên hóa có hiệu lực'),
    ('Sai lệch ước lượng', f'Ngây thơ {ate_naive*100:.2f} pp vs thật {ate_rct*100:.2f} pp '
                           f'-> thổi phồng {ate_naive/ate_rct:.1f} lần'),
    ('Dịch chuyển quan sát vs RCT', f'{(shift > 0.25).sum()}/{n_feat} đặc trưng lệch > 0,25 -> nhẹ hơn thiên vị treatment'),
    ('Tính không đồng nhất', 'Uplift chênh 1,6–2,7 lần ATE giữa các phân khúc -> bài toán CATE chính đáng'),
], columns=['Hạng mục', 'Kết quả'])

findings

### Bước 43 — Ghi file bàn giao cho notebook 02

In [ ]:
eda_output = {
    'so_dong': {'train': int(len(train)), 'rct_select': int(len(rct))},
    'cot_hang_so': constant_cols,
    'cot_trung_lap': duplicate_map,
    'cot_can_loai_bo': COLS_TO_DROP,
    'cot_thua_tuyen_tinh': LINEAR_REDUNDANT,
    'dac_trung_sach': CLEAN_FEATURES,
    'top20_thien_vi': smd_train.head(20).index.tolist(),
    'chi_so_chinh': {
        'ti_le_treat_train': float(train.is_treat.mean()),
        'ti_le_treat_rct_select': float(rct.is_treat.mean()),
        'cvr_train': float(train.label.mean()),
        'cvr_rct_select': float(rct.label.mean()),
        'ate_ngay_tho': float(ate_naive),
        'ate_rct': float(ate_rct),
        'ate_rct_ci95': [float(ci_lo), float(ci_hi)],
        'ty_le_thoi_phong': float(ate_naive / ate_rct),
        'smd_max_train': float(smd_train.max()),
        'smd_max_test': float(smd_rct.max()),
        'so_dac_trung_smd_lon_025': int((smd_train > 0.25).sum()),
    },
    'khuyen_nghi': [
        'Chia dữ liệu phải stratify theo (is_treat, label) vì positive chỉ ~2%',
        'Dùng importance_type="gain" khi tính feature importance, vì gần nửa số cột là nhị phân',
        'Không cần chuẩn hóa scale — toàn bộ mô hình dùng cây quyết định',
        'Cắt propensity score trong khoảng [0.01, 0.99] để tránh trọng số bùng nổ',
        'Chỉ đo Qini/AUUC trên tập RCT; trên dữ liệu quan sát phải dùng DR-score',
    ],
}

out_path = os.path.join(ART_DIR, 'eda_columns_to_drop.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(eda_output, f, ensure_ascii=False, indent=2)

print('Đã ghi:', out_path)
print('Số biểu đồ đã lưu:', len([x for x in os.listdir(FIG_DIR) if x.endswith('.png')]))

---
## Kết luận

### Ba kết quả quan trọng nhất

**1. Dữ liệu quan sát thiên vị nặng.** Phần lớn đặc trưng vượt xa ngưỡng lệch nghiêm trọng giữa hai nhóm, trong khi tập RCT gần như cân bằng hoàn toàn. Đây là bằng chứng trực tiếp cho việc phải dùng phương pháp nhân quả thay vì mô hình dự đoán thông thường.

**2. Ước lượng ngây thơ sai lệch hơn một chục lần.** Con số thuyết phục nhất để đưa vào slide, vì nó cho thấy hậu quả cụ thể của việc bỏ qua thiên vị chọn lọc.

**3. Hiệu ứng không đồng nhất giữa các phân khúc.** Chỉ dùng thống kê mô tả trên một đặc trưng đã thấy phân hóa rõ, chứng minh bài toán ước lượng CATE là chính đáng.

### Những gì quyết định cho các bước sau

| Phát hiện | Ảnh hưởng tới |
|---|---|
| 0 giá trị khuyết thiếu | Notebook 02 bỏ qua khâu điền khuyết |
| Danh sách cột trùng lặp và hằng số | Notebook 02 loại bỏ, còn lại tập đặc trưng sạch |
| Gần nửa số cột là nhị phân | Notebook 04 phải dùng `importance_type='gain'` |
| Thang đo các cột rất khác nhau | Không cần chuẩn hóa vì dùng mô hình cây |
| Chỉ ~2% mua hàng | Chia dữ liệu bắt buộc stratify theo `(is_treat, label)` |
| Hai nhóm vẫn chồng lấn | Positivity còn giữ được, notebook 02 xác nhận bằng propensity score |
| Tập Train có bias | Không đo Qini trên Val được, phải dùng DR-score |

### Notebook tiếp theo

`02_feature_engineering.ipynb` — loại cột rác theo danh sách vừa ghi ra, tạo đặc trưng dẫn xuất, kiểm tra positivity bằng propensity score, và chia dữ liệu thành 4 tập:

```
full_trainset  ->  Train 80%       (huấn luyện)
                   Val   20%       (tinh chỉnh siêu tham số)

full_testset   ->  RCT-select  50% (chọn quán quân giữa các mô hình)
                   RCT-holdout 50% (chạy một lần, ra số báo cáo cuối)
```